# Get Kegg pahtway statistics
collect statistics for the kegg pathways selevted in "get_kegg_pw_nodes". Store the metrics in data_phase1 and use them to select suitable pathways for future use

In [1]:
import pandas as pd
parent_dir = "data_phase1"
df_cell_info = pd.read_csv(f'{parent_dir}/cell_info.txt', sep='\t')     # Metadata for cell lines (the samples of the data matrices)
df_gene_info = pd.read_csv(f'{parent_dir}/gene_info.txt', sep='\t')     # Metadata for genes (the features of the data matrices)
df_inst_info = pd.read_csv(f'{parent_dir}/inst_info.txt', sep='\t')     # Metadata for L1000 experiments (the data matrices)
df_pert_info = pd.read_csv(f'{parent_dir}/pert_info.txt', sep='\t')     # Metadata for perturbations applied in L1000 experiments
df_sig_info = pd.read_csv(f'{parent_dir}/sig_info.txt', sep='\t')       # Metadata for level 5 profiles

/tmp/ipykernel_1063511/443091508.py:5: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_inst_info = pd.read_csv(f'{parent_dir}/inst_info.txt', sep='\t')     # Metadata for L1000 experiments (the data matrices)
/tmp/ipykernel_1063511/443091508.py:7: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_sig_info = pd.read_csv(f'{parent_dir}/sig_info.txt', sep='\t')       # Metadata for level 5 profiles


In [2]:
# join df_sig_info with df gene_info to get gene info on the perturbed genes
df_perturbed_genes = df_sig_info.merge(df_gene_info, left_on='pert_iname', right_on='pr_gene_symbol', how='inner')#
unique_perturbed_genes = df_perturbed_genes['pert_iname'].unique()

In [5]:
import json
import os

parent_dir = "kegg_data"

# read kegg_nodes.json
kegg_nodes_path = os.path.join(parent_dir, "kegg_nodes.json")
with open(kegg_nodes_path, "r", encoding="utf-8") as fin:
    kegg_nodes = json.load(fin)

metrics = {}
unique_perturbed_set = set(unique_perturbed_genes)
for pathway, dict in kegg_nodes.items():
    nodes = dict.get('nodes', [])
    num_edges = dict.get('edges', 0)
    if not isinstance(nodes, (list, tuple, set)):
        raise ValueError(f"nodes for pathway {pathway!r} must be a list-like object")
    nodes_list = list(nodes)
    num_nodes = len(nodes_list)

    # directed adjacency without self-loops as in prior code (each ordered pair counted)
    avg_degree = num_edges / num_nodes
    density = num_edges / (num_nodes * num_nodes)

    num_perturbed_in_pathway = sum(1 for node in nodes_list if node in unique_perturbed_set)
    node_set = set(nodes_list)
    num_sig_info_in_pathway = int(df_sig_info['pert_iname'].isin(node_set).sum())

    metrics[pathway] = {
        'number_of_nodes': int(num_nodes),
        'number_of_edges': int(num_edges),
        'average_degree': float(avg_degree),
        'density': float(density),
        'number_of_perturbed_genes_in_pathway': int(num_perturbed_in_pathway),
        'number_of_sig_info_in_pathway': int(num_sig_info_in_pathway)
    }

out_path = os.path.join(parent_dir, "kegg_nodes_metrics.json")
with open(out_path, "w", encoding="utf-8") as fout:
    json.dump(metrics, fout, indent=4, ensure_ascii=False)

print(f"Wrote metrics for {len(metrics)} pathways to {out_path}")

Wrote metrics for 17 pathways to kegg_data/kegg_nodes_metrics.json
